In [1]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_V2_S_Weights

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

def get_peak_memory():
    return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
csv_file = "results/EfficientNetV2s_metrics.csv"


if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])

In [3]:
# Pretrained EfficientNetV2-S weights
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1


# Use simple preprocessing only
transform = weights.transforms()

# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 68175
Val samples: 7575
Test samples: 25250


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

method_name = "efficientnet_adapter_finetune"

# -------------------------
# Small adapter module
# -------------------------
class TSA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden_dim = max(1, channels // reduction)

        self.adapter = nn.Sequential(
            nn.Conv2d(channels, hidden_dim, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_dim, channels, kernel_size=1)
        )

    def forward(self, x):
        return x + self.adapter(x)  # residual connection


# -------------------------
# EfficientNetV2-S + TSA
# -------------------------
class EfficientNetV2S_TSA(nn.Module):
    def __init__(self, num_classes=101, reduction=16,
                 weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1):
        super().__init__()

        base = efficientnet_v2_s(weights=weights)

        # Stem
        self.features = self._add_adapters(base.features, reduction)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(base.classifier[1].in_features, num_classes)

    def _add_adapters(self, features, reduction):
        new_layers = []

        for layer in features:
            if isinstance(layer, nn.Sequential):
                blocks = []
                for block in layer:
                    # Try to infer output channels
                    if hasattr(block, "out_channels"):
                        channels = block.out_channels
                    elif hasattr(block, "conv"):
                        channels = block.conv.out_channels
                    else:
                        # fallback (works for most torchvision blocks)
                        channels = block[-1].out_channels if isinstance(block, nn.Sequential) else None

                    if channels is None:
                        blocks.append(block)
                    else:
                        adapter = TSA(channels, reduction)
                        blocks.append(nn.Sequential(block, adapter))

                new_layers.append(nn.Sequential(*blocks))
            else:
                new_layers.append(layer)

        return nn.Sequential(*new_layers)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# -------------------------
# Load pretrained model
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

model = EfficientNetV2S_TSA(num_classes=101).to(device)


# -------------------------
# Freeze everything
# -------------------------
for param in model.parameters():
    param.requires_grad = False


# -------------------------
# Unfreeze ONLY adapters
# -------------------------
for module in model.modules():
    if isinstance(module, TSA):
        for p in module.parameters():
            p.requires_grad = True


# -------------------------
# Unfreeze classifier
# -------------------------
for param in model.classifier.parameters():
    param.requires_grad = True


# -------------------------
# Training setup
# -------------------------
print("Using device:", device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

Using device: cuda


In [5]:
criterion = nn.CrossEntropyLoss()

## optimizeer for the batch norm and the classifier
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [6]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print(f"Time: {epoch_time:.2f}s | Peak Mem: {peak_memory:.2f} MB")
    print("-" * 40)

    # SAVE METRICS TO CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            method_name,
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            peak_memory,
            trainable_params
        ])

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        model_path = f"models/efficientnet_tsa_finetune_best.pth"
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "epoch": epoch
        }, model_path)

        print("Best model saved!")

Trainable params: 509,830
Epoch 1/8
Train Loss: 1.5340, Train Acc: 0.6055
Val Loss:   0.9259, Val Acc:   0.7529
Time: 5107.36s | Peak Mem: 7571.99 MB
----------------------------------------
Best model saved!
Epoch 2/8
Train Loss: 0.9641, Train Acc: 0.7414
Val Loss:   0.8434, Val Acc:   0.7725
Time: 5071.71s | Peak Mem: 7563.46 MB
----------------------------------------
Best model saved!
Epoch 3/8
Train Loss: 0.8277, Train Acc: 0.7745
Val Loss:   0.7854, Val Acc:   0.7914
Time: 5248.61s | Peak Mem: 7563.46 MB
----------------------------------------
Best model saved!
Epoch 4/8
Train Loss: 0.7481, Train Acc: 0.7940
Val Loss:   0.7612, Val Acc:   0.8022
Time: 5184.20s | Peak Mem: 7563.46 MB
----------------------------------------
Best model saved!
Epoch 5/8
Train Loss: 0.6946, Train Acc: 0.8080
Val Loss:   0.7433, Val Acc:   0.8032
Time: 5111.15s | Peak Mem: 7563.46 MB
----------------------------------------
Best model saved!
Epoch 6/8
Train Loss: 0.6446, Train Acc: 0.8208
Val Loss:  